In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import random
import mlflow

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score, f1_score, brier_score_loss, auc
from sklearn.preprocessing import StandardScaler, label_binarize, LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from mlflow.tracking import MlflowClient


import xgboost as xgb

from itertools import product

pd.set_option('display.max_rows', 1000)

In [ ]:
# Definition of function which produce confusion matrix for train and also val set 
def class_report(y_train, y_train_pred, y_val, y_val_pred, target_name):
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 6), constrained_layout=True) 
    
    # Train Confusion Matrix
    ConfusionMatrixDisplay.from_predictions(
        y_train, y_train_pred, 
        ax=axes[0], cmap='Blues', normalize='true',
        labels=target_name, values_format='.2f',
        colorbar=False  
    )
    axes[0].set_title("Training Confusion Matrix (Normalized)", fontsize=16, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].tick_params(axis='y', rotation=45)

    axes[0].set_xlabel("Predicted label", fontsize=14)
    axes[0].set_ylabel("True label", fontsize=14)
    
    # Val Confusion Matrix
    ConfusionMatrixDisplay.from_predictions(
        y_val, y_val_pred, 
        ax=axes[1], cmap='Greens', normalize='true',
        labels=target_name, values_format='.2f',
        colorbar=False  
    )
    axes[1].set_title("Validation Confusion Matrix (Normalized)", fontsize=16, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].tick_params(axis='y', rotation=45) 

    axes[1].set_xlabel("Predicted label", fontsize=14)
    axes[1].set_ylabel("True label", fontsize=14)
    
    # Save as high-resolution PNG
    plt.savefig("./plots/class_report.png", format='png', dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# Definition of function which produce calibration curve per each region for train and also val set 
def calibration_curve_classes(y_train, y_train_proba, y_train_pred, y_val, y_val_proba, y_val_pred, colors, labels):
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    
    for j, a in zip(range(2), ["Train", "Val"]):
        ax[j].plot([0, 1], [0, 1], "k--", label="Perfectly Calibrated")
        ax[j].plot([0.75, 0.75], [-0.05, 1.05], "--", color="#C0C0C0")
        for l, c in zip(labels, colors):
            proba = locals()[f"y_{a.lower()}_proba"]
            pred  = locals()[f"y_{a.lower()}_pred"]
            true  = locals()[f"y_{a.lower()}"]
        
            subset = (true == l)
            
            confidences = np.max(proba[subset], axis=1)
            accuracies = (pred[subset] == true[subset]).astype(int)
            n_bins = 5
            percentiles = np.linspace(0.0, 1.0, n_bins + 1)
            bins = np.quantile(confidences, percentiles)
            bin_indices = np.digitize(confidences, bins) - 1
            bin_indices[bin_indices == n_bins] = n_bins - 1
            
            bin_accuracies = []
            bin_confidences = []
            bin_percentages = []
            
            for i in range(n_bins):
                mask = (bin_indices == i)
                if np.sum(mask) > 0: # Ak sú v bine nejaké dáta
                    bin_accuracies.append(np.mean(accuracies[mask]))
                    bin_confidences.append(np.mean(confidences[mask]))
                    bin_percentages.append((np.sum(mask) / len(confidences)) * 100)
        
            ax[j].plot(bin_confidences, bin_accuracies, marker='o', linestyle='-', color=c, label=l)
        
        ax[j].set_xlabel("Confidence", fontsize=14)
        ax[j].set_ylabel("True Accuracy", fontsize=14)
        ax[j].set_title(f"Calibration Graph({a} Set) ", fontsize=16, fontweight='bold')
        ax[j].legend(loc="upper left")
        ax[j].grid(True, alpha=0.5)
        ax[j].set_xlim([-0.05, 1.05])
        ax[j].set_ylim([-0.05, 1.05])
        
    plt.savefig("./plots/calibration_graph.png", dpi=300, bbox_inches='tight')
    plt.close()

In [16]:
def calculate_aurc(y_true, y_prob, y_pred):
    """
    Calculates the Area Under the Risk-Coverage Curve (AURC).
    """
    # 1. Convert to pure NumPy arrays to avoid Pandas KeyErrors
    y_true_np = np.array(y_true)
    y_pred_np = np.array(y_pred)
    
    # 2. Extract the model's confidence
    confidences = np.max(y_prob, axis=1)
    
    # 3. Determine which predictions are incorrect (Error = 1, Correct = 0)
    # By using y_pred directly, this safely compares strings to strings or ints to ints!
    errors = (y_true_np != y_pred_np).astype(int)
    
    # 4. Sort the samples by confidence in descending order
    sorted_indices = np.argsort(-confidences)
    sorted_errors = errors[sorted_indices]
    
    # 5. Calculate Coverage (x-axis)
    n_samples = len(y_true_np)
    coverages = np.arange(1, n_samples + 1) / n_samples
    
    # 6. Calculate Risk (y-axis): The cumulative mean of errors
    cumulative_errors = np.cumsum(sorted_errors)
    risks = cumulative_errors / np.arange(1, n_samples + 1)
    
    # 7. Add the starting point (0 coverage) to anchor the graph
    coverages = np.insert(coverages, 0, 0.0)
    risks = np.insert(risks, 0, risks[0]) 
    
    
    return coverages, risks

In [ ]:
def plot_risk_coverage_curve(y_train, y_train_proba, y_train_pred, y_val, y_val_proba, y_val_pred):
    """Plots the Risk-Coverage curves for Train and Val sets side-by-side."""
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Bundle the datasets to iterate through them easily
    datasets = [
        ("Train", y_train, y_train_proba, y_train_pred, axes[0]),
        ("Val", y_val, y_val_proba, y_val_pred, axes[1])
    ]
    
    for name, true, proba, pred, ax in datasets:
        # Get curve coordinates and score (Passing 'pred' here to fix the string mismatch)
        coverages, risks = calculate_aurc(true, proba, pred)
        
        # Plot the actual curve
        ax.plot(coverages, risks, color='blue', linewidth=2, label=f'Model AURC')
        
        # Plot the baseline (Full dataset error)
        # Converted to np.array to prevent Pandas index mismatches here as well
        baseline_error = np.mean(np.array(true) != np.array(pred))
        ax.axhline(y=baseline_error, color='red', linestyle='--', label=f'Full Dataset Error ({baseline_error:.4f})')
        
        # Formatting
        ax.set_title(f"Risk-Coverage Curve ({name} Set)", fontsize=16, fontweight='bold')
        ax.set_xlabel("Coverage (Proportion of dataset evaluated)", fontsize=14)
        ax.set_ylabel("Risk (Error rate on covered samples)", fontsize=14)
        ax.legend(loc="upper left")
        ax.grid(True, alpha=0.5)
        ax.set_xlim([-0.05, 1.05])
        
    plt.savefig("./plots/risk_coverage.png", dpi=300, bbox_inches="tight")
    plt.close()

In [18]:
def get_mlflow_confidence_metrics(
    y_val_true, y_val_pred, y_val_prob, 
    y_train_true, y_train_pred, y_train_prob, 
    target_names
):
    metrics = {}

    metrics['train_brier_score'] = float(brier_score_loss(label_binarize(y_train_true, classes=target_names), y_train_prob))
    metrics['val_brier_score'] = float(brier_score_loss(label_binarize(y_val_true, classes=target_names), y_val_prob))
    metrics['train_accuracy'] = float(accuracy_score(y_train_true, y_train_pred))
    metrics['val_accuracy'] = float(accuracy_score(y_val_true, y_val_pred))
    metrics['samples'] = len(y_val_true)

    
    # 3. Confidence Thresholds (Calculated on the Validation set)
    y_val_true = np.array(y_val_true)
    y_val_pred = np.array(y_val_pred)
    confidences = np.max(np.array(y_val_prob), axis=1)
    
    thresholds = [0.50, 0.60, 0.70, 0.80, 0.90]
    
    for t in thresholds:
        t_str = f"val_conf_{int(t * 100)}"
        mask = (confidences >= t)
        n_samples = int(np.sum(mask))
        
        metrics[f"{t_str}_samples"] = n_samples
        if n_samples > 0:
            metrics[f"{t_str}_accuracy"] = float(accuracy_score(y_val_true[mask], y_val_pred[mask]))
        else:
            metrics[f"{t_str}_accuracy"] = 0.0
            
    return metrics

In [19]:
#Importing data from csv, the path could be different
data = pd.read_csv("/home/matej/btc_timezone_analysis/data_preprocessing/database_from_bitcointalk.csv")
len(data)
data.columns = data.columns.astype(str)
data.head()
X = data
y = data["region"]
col_names = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "20", "21", "22", "23"]

In [20]:
#Preparing test and train set based on selected random state
num = 1019
print(num)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify = y, random_state = num)

#Creating sample weights, which put higher weight on smaller classes and lower weight on bigger classes 
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

1019


In [25]:
target_names = ["America", "Euro_Africa", "Central_South_Asia", "East_Asia_Pac"]
colors = ['red', 'green', 'blue', 'black']


pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('log_reg', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

kf = StratifiedKFold(n_splits=5, shuffle=True)

y_val_cv = []
y_val_cv_pred = []
y_val_cv_proba = []

y_train_cv = []
y_train_cv_pred = []
y_train_cv_proba = []


for train_idx, val_idx in kf.split(X_train[col_names], y_train):
    X_tr, X_va = X_train[col_names].iloc[train_idx], X_train[col_names].iloc[val_idx]
    y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train the pipeline
    pipeline.fit(X_tr, y_tr)

    # Predict and store Training data
    y_train_cv.extend(y_tr)
    y_train_cv_pred.extend(pipeline.predict(X_tr))
    y_train_cv_proba.extend(pipeline.predict_proba(X_tr))
    
    # Predict and store Validation data
    y_val_cv.extend(y_va)
    y_val_cv_pred.extend(pipeline.predict(X_va))
    y_val_cv_proba.extend(pipeline.predict_proba(X_va))

# Convert training lists to numpy arrays
y_train_cv = np.array(y_train_cv)
y_train_cv_pred = np.array(y_train_cv_pred)
y_train_cv_proba = np.array(y_train_cv_proba)
y_val_cv = np.array(y_val_cv)
y_val_cv_pred = np.array(y_val_cv_pred)
y_val_cv_proba = np.array(y_val_cv_proba)

metrics = get_mlflow_confidence_metrics(
    y_val_cv, y_val_cv_pred, y_val_cv_proba, 
    y_train_cv, y_train_cv_pred, y_train_cv_proba, 
    pipeline.classes_
)

class_report(y_train_cv, y_train_cv_pred, y_val_cv, y_val_cv_pred, target_names)
calibration_curve_classes(y_train_cv, y_train_cv_proba, y_train_cv_pred, y_val_cv, y_val_cv_proba, y_val_cv_pred, colors, target_names)
plot_risk_coverage_curve(y_train_cv, y_train_cv_proba, y_train_cv_pred, y_val_cv, y_val_cv_proba, y_val_cv_pred)

